# Notebook 7 — Supervised departure prediction

Binary classification: will a resident leave Lausanne? Models are trained on
**entry features only** (information available when the individual enters
observation). Trajectory features are excluded from the honest models and
used once, deliberately, to quantify target leakage. Population: the NB6
enriched analytical table, modelling perimeter of 298,421 individuals
(deaths and pending departures excluded).

Block map: 0 — population, feature matrix, split · 1 — entry-only baseline ·
2 — leakage quantification · 3 — full entry feature set (logistic) ·
4 — tree ensembles · 4b — exposure artifact bound · 5 — final table and
figures · 6a/6b — error analysis and interaction
detection (the spatial contingency of profile clusters, formerly 6c, now
lives at the end of NB8, after the clusters it consumes are built).

## Block 0 — Modelling population, feature matrix and train/test split

The enriched analytical table (NB6) is merged with the train/test split
(`nb7_model_population.parquet`). The split file is **created here on first
run** (perimeter = deaths and pending departures excluded; stratified 75/25
split, seed 42) and reused afterwards, so that all blocks and later re-runs
share the exact same test set and metrics remain comparable. Regenerating it
(e.g. after a corpus update: delete the file) resets that comparability —
expected and documented. The perimeter exclusions are inherited through the
inner merge. The table is sorted by individual identifier so that all blocks
operate on identically ordered rows.

The district of the first episode is attached via the EGID -> district
correspondence (17 districts plus special code 90; 60 unmatched EGIDs are
kept as 'inconnu'). Missing numerical features are imputed with the
training-set median only (no test leakage), with missingness indicator flags
kept as features (an unknown dwelling is itself informative).

The resulting entry feature matrix has 54 columns: entry type, entry year,
born-resident flag, district, plus the demographic enrichment (age, sex,
nationality group, permit group, provenance group, household size and type,
dwelling size).

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("/mnt/o/09_STATISTIQUES/09_04_Explorations/MR/CAS ADS/Final project/data")
FIG_DIR = DATA_DIR.parent / "figures"
RANDOM_STATE = 42

# Parallelism cap. With n_jobs=-1, cross-validation spawns one worker process
# per logical core, EACH holding a pickled copy of the training matrix; under
# WSL's default memory limit this OOM-kills the Jupyter server (observed at
# Block 3, where the matrix is 5x wider than in Blocks 1-2). Two folds in
# parallel is plenty; tree ensembles parallelise internally instead.
N_JOBS_CV = 2

# --- Enriched table + train/test split (created on first run, then reused) ---
pers = pd.read_parquet(DATA_DIR / "persons_analytical_enriched.parquet")

SPLIT_PATH = DATA_DIR / "nb7_model_population.parquet"
if not SPLIT_PATH.exists():
    from sklearn.model_selection import train_test_split
    base = pers[~pers["depart_status"].isin(["death", "pending_anticipated"])].copy()
    base["annee_debut"] = base["date_debut_premier"].dt.year
    base["ne_et_reside_depuis_naissance"] = base["ne_et_reside_depuis_naissance"].astype(int)
    tr_ids, _ = train_test_split(base["id_projet"], test_size=0.25,
                                 stratify=base["target_depart"],
                                 random_state=RANDOM_STATE)
    base["split"] = np.where(base["id_projet"].isin(set(tr_ids)), "train", "test")
    base[["id_projet", "split", "target_depart", "annee_debut",
          "debut_type_premier", "ne_et_reside_depuis_naissance",
          "n_episodes", "duree_totale_j", "duree_median_ep_j", "n_absences"]] \
        .to_parquet(SPLIT_PATH, index=False)
    print(f"Created {SPLIT_PATH.name}: {len(base):,} rows "
          f"(train {(base['split'] == 'train').sum():,} / "
          f"test {(base['split'] == 'test').sum():,})")

split = pd.read_parquet(SPLIT_PATH)[["id_projet", "split"]]
pop = (pers.merge(split, on="id_projet", how="inner")     # inner = NB7 perimeter
           .sort_values("id_projet").reset_index(drop=True))
print(f"Modelling population: {len(pop):,}")
print(f"Departure rate: {pop['target_depart'].mean():.1%}")
assert len(pop) == len(split), \
    "Perimeter mismatch between enriched table and split file -- delete the split file and re-run"

pop["annee_debut"] = pop["date_debut_premier"].dt.year
pop["ne_et_reside_depuis_naissance"] = pop["ne_et_reside_depuis_naissance"].astype(int)

# --- District of the first episode (EGID -> quartier) ---
episodes = pd.read_parquet(DATA_DIR / "episodes.parquet", columns=["id_projet", "ep_num", "loc"])
first_ep = episodes.sort_values(["id_projet", "ep_num"]).groupby("id_projet", as_index=False).first()
first_ep["egid"] = first_ep["loc"].str.split("-").str[0]
adr = pd.read_excel(DATA_DIR.parent / "Adresses.xlsx", sheet_name="Egid")[["EGID", "NUMQUARTIER"]] \
        .rename(columns={"NUMQUARTIER": "quartier"}).drop_duplicates("EGID")
adr["EGID"] = adr["EGID"].astype(str); adr["quartier"] = adr["quartier"].astype(str)
first_ep = first_ep.merge(adr, left_on="egid", right_on="EGID", how="left")
pop = pop.merge(first_ep[["id_projet", "quartier"]], on="id_projet", how="left")
pop["quartier"] = pop["quartier"].fillna("inconnu")

# --- Imputation of numerics (median from TRAIN only + missingness flags) ---
mask_train = (pop["split"] == "train").values
for c in ["age_entree", "menage_taille", "pieces", "surface"]:
    pop[f"{c}_manquant"] = pop[c].isna().astype(int)
    med = pop.loc[mask_train, c].median()
    pop[c] = pop[c].fillna(med)
print("\nImputed (train medians):",
      {c: round(pop.loc[mask_train, c].median(), 1)
       for c in ["age_entree", "menage_taille", "pieces", "surface"]})

# --- Enriched entry feature matrix (54 columns) ---
CAT = ["debut_type_premier", "quartier", "sexe", "nat_groupe",
       "permis_groupe", "prov_groupe", "menage_type"]
NUM = ["annee_debut", "ne_et_reside_depuis_naissance", "age_entree",
       "menage_taille", "pieces", "surface",
       "pieces_manquant", "surface_manquant", "age_entree_manquant"]
X_enr = pd.get_dummies(pop[CAT + NUM], columns=CAT, dtype=int).astype("float32")
ENTRY_ENR_COLS = list(X_enr.columns)
y_all = pop["target_depart"]
X_train_en, X_test_en = X_enr[mask_train], X_enr[~mask_train]
y_train, y_test = y_all[mask_train], y_all[~mask_train]
print(f"\nFeature matrix: {X_enr.shape[0]:,} x {X_enr.shape[1]} "
      f"(train {mask_train.sum():,} / test {(~mask_train).sum():,})")

# Memory hygiene: episode/address intermediates are no longer needed
import gc
del episodes, first_ep, adr
_ = gc.collect()

## Block 1 — Baseline: logistic regression on entry features only

The baseline uses the original coarse entry features (first entry type,
entry year, born-and-resident flag): information available at entry, free of
target leakage by construction, setting the reference performance level.
Numerical features are standardised inside a pipeline (scaler fitted on the
training data only). Classes are nearly balanced (50.1% departures), so a
majority-class baseline sits at ~50% accuracy. Uncertainty is estimated by
5-fold stratified cross-validation on the training set.

This block reads the original modelling table into `pop_base` (distinct name:
it must NOT overwrite the enriched `pop` from Block 0) and verifies that both
tables are identically ordered, so that predicted probabilities from all
blocks refer to the same test rows (required for the joint ROC figure).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

# --- Constants for the baseline feature set (self-contained block) ---
FEATURES_ENTRY_CAT = ["debut_type_premier"]
FEATURES_ENTRY_NUM = ["annee_debut", "ne_et_reside_depuis_naissance"]
FEATURES_TRAJ = ["n_episodes", "duree_totale_j", "duree_median_ep_j", "n_absences"]

pop_base = (pd.read_parquet(DATA_DIR / "nb7_model_population.parquet")
              .sort_values("id_projet").reset_index(drop=True))
# Alignment guards: same individuals, same order, same split as Block 0
assert (pop_base["id_projet"].values == pop["id_projet"].values).all()
assert ((pop_base["split"] == "train").values == mask_train).all()

X_all = pd.get_dummies(
    pop_base[FEATURES_ENTRY_CAT + FEATURES_ENTRY_NUM + FEATURES_TRAJ],
    columns=FEATURES_ENTRY_CAT, prefix="entree", dtype=int,
).astype("float32")
ENTRY_COLS = [c for c in X_all.columns if c.startswith("entree_")] + FEATURES_ENTRY_NUM

X_train_e = X_all.loc[mask_train, ENTRY_COLS]
X_test_e  = X_all.loc[~mask_train, ENTRY_COLS]

logit_entry = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_res = cross_validate(logit_entry, X_train_e, y_train, cv=cv,
                        scoring=["roc_auc", "accuracy"], n_jobs=N_JOBS_CV)
print("5-fold CV (train):")
print(f"  ROC-AUC : {cv_res['test_roc_auc'].mean():.4f} +/- {cv_res['test_roc_auc'].std():.4f}")
print(f"  Accuracy: {cv_res['test_accuracy'].mean():.4f} +/- {cv_res['test_accuracy'].std():.4f}")

logit_entry.fit(X_train_e, y_train)
y_pred  = logit_entry.predict(X_test_e)
y_proba = logit_entry.predict_proba(X_test_e)[:, 1]

metrics_entry = {
    "model": "logit_entry",
    "accuracy":  accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred),
    "recall":    recall_score(y_test, y_pred),
    "f1":        f1_score(y_test, y_pred),
    "roc_auc":   roc_auc_score(y_test, y_proba),
}
print("\nTest set performance (entry features only):")
for k, v in metrics_entry.items():
    if k != "model":
        print(f"  {k:9s}: {v:.4f}")

coefs = pd.Series(logit_entry.named_steps["clf"].coef_[0],
                  index=ENTRY_COLS).sort_values(key=abs, ascending=False)
print("\nStandardised coefficients (sorted by |value|):")
print(coefs.astype(float).round(3).to_string())

all_results = [metrics_entry]

## Block 2 — Adding trajectory features: quantifying the leakage

The same logistic pipeline is retrained with the trajectory features added
(number of episodes, total and median episode duration, number of absences).
These features accumulate over the observation window, whose length depends
on the outcome itself: a censored (present) individual is observed until the
end of the window, while a departed individual stops accumulating at
departure. The performance gap between this model and the entry-only
baseline therefore measures a mix of genuine signal and partial target
leakage, and is reported as such rather than as a pure improvement. The
mechanism is visible in the coefficients: entry year plus total duration
approximately reconstruct the end of the observation window, revealed by the
sign inversion and magnitude of the entry-year coefficient.

In [ ]:
FULL_COLS = ENTRY_COLS + FEATURES_TRAJ
X_train_f = X_all.loc[mask_train, FULL_COLS]
X_test_f  = X_all.loc[~mask_train, FULL_COLS]

logit_full = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

cv_res_f = cross_validate(logit_full, X_train_f, y_train, cv=cv,
                          scoring=["roc_auc", "accuracy"], n_jobs=N_JOBS_CV)
print("5-fold CV (train, entry + trajectory):")
print(f"  ROC-AUC : {cv_res_f['test_roc_auc'].mean():.4f} +/- {cv_res_f['test_roc_auc'].std():.4f}")
print(f"  Accuracy: {cv_res_f['test_accuracy'].mean():.4f} +/- {cv_res_f['test_accuracy'].std():.4f}")

logit_full.fit(X_train_f, y_train)
y_pred_f  = logit_full.predict(X_test_f)
y_proba_f = logit_full.predict_proba(X_test_f)[:, 1]

metrics_full = {
    "model": "logit_full_leaky",
    "accuracy":  accuracy_score(y_test, y_pred_f),
    "precision": precision_score(y_test, y_pred_f),
    "recall":    recall_score(y_test, y_pred_f),
    "f1":        f1_score(y_test, y_pred_f),
    "roc_auc":   roc_auc_score(y_test, y_proba_f),
}
print("\nTest set performance (entry + trajectory):")
for k, v in metrics_full.items():
    if k != "model":
        print(f"  {k:9s}: {v:.4f}")
print(f"\nDelta ROC-AUC vs entry-only baseline: "
      f"+{metrics_full['roc_auc'] - metrics_entry['roc_auc']:.4f}")

coefs_f = pd.Series(logit_full.named_steps["clf"].coef_[0],
                    index=FULL_COLS).sort_values(key=abs, ascending=False)
print("\nStandardised coefficients (sorted by |value|):")
print(coefs_f.astype(float).round(3).to_string())

all_results.append(metrics_full)

## Block 3 — Logistic regression on the full entry feature set

The logistic pipeline is retrained on the 54-column entry matrix from
Block 0 (demographic + household + district + entry type). This is the
honest linear model: every feature is known at entry, none derives from the
observation window. The guard assertion protects against the matrix being
silently overwritten by another block (a failure mode encountered during
development).

In [ ]:
# Guard: the matrix must be the demographic one built in Block 0
assert X_train_en.shape[1] == 54, \
    f"Expected 54 features from Block 0, got {X_train_en.shape[1]} — run Block 0 first"
print(f"Training on {X_train_en.shape[1]} entry features "
      f"(demographic + household + district + entry type)")

logit_enr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])

cv_res_en = cross_validate(logit_enr, X_train_en, y_train, cv=cv,
                           scoring=["roc_auc", "accuracy"], n_jobs=N_JOBS_CV)
print("\n5-fold CV (train, full entry features):")
print(f"  ROC-AUC : {cv_res_en['test_roc_auc'].mean():.4f} +/- {cv_res_en['test_roc_auc'].std():.4f}")
print(f"  Accuracy: {cv_res_en['test_accuracy'].mean():.4f} +/- {cv_res_en['test_accuracy'].std():.4f}")

logit_enr.fit(X_train_en, y_train)
y_pred_en  = logit_enr.predict(X_test_en)
y_proba_en = logit_enr.predict_proba(X_test_en)[:, 1]

metrics_enr = {
    "model": "logit_entry_demographic",
    "accuracy":  accuracy_score(y_test, y_pred_en),
    "precision": precision_score(y_test, y_pred_en),
    "recall":    recall_score(y_test, y_pred_en),
    "f1":        f1_score(y_test, y_pred_en),
    "roc_auc":   roc_auc_score(y_test, y_proba_en),
}
print("\nTest set performance (full entry features):")
for k, v in metrics_enr.items():
    if k != "model":
        print(f"  {k:9s}: {v:.4f}")

coefs_en = pd.Series(logit_enr.named_steps["clf"].coef_[0],
                     index=ENTRY_ENR_COLS).sort_values(key=abs, ascending=False)
print("\nStandardised coefficients (top 15 by |value|):")
print(coefs_en.astype(float).head(15).round(3).to_string())

all_results.append(metrics_enr)

## Block 4 — Tree-based models on the full entry feature set

Random forest and histogram gradient boosting on the same 54 entry features.
Tree ensembles capture non-linear effects and interactions (e.g. district x
permit, age x entry year) that the logistic model cannot. Hyperparameters
are kept close to defaults with light regularisation (min_samples_leaf=50 on
the forest to avoid overfitting on one-hot columns); the goal is a fair
comparison of model families within the project scope, not exhaustive
tuning. Impurity-based importances mechanically favour continuous variables
(entry year, age), a known limitation mentioned where they are reported.

In [ ]:
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

models = {
    "random_forest_demographic": RandomForestClassifier(
        n_estimators=200, min_samples_leaf=50,
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "hgb_demographic": HistGradientBoostingClassifier(
        random_state=RANDOM_STATE,
    ),
}

for name, model in models.items():
    # n_jobs=1 on the folds: the models already use every core internally
    # (RF n_jobs=-1, HGB via OpenMP); parallel folds on top would multiply
    # both memory and thread count (nested parallelism).
    cv_r = cross_validate(model, X_train_en, y_train, cv=cv,
                          scoring=["roc_auc", "accuracy"], n_jobs=1)
    print(f"\n{name} — 5-fold CV (train):")
    print(f"  ROC-AUC : {cv_r['test_roc_auc'].mean():.4f} +/- {cv_r['test_roc_auc'].std():.4f}")
    print(f"  Accuracy: {cv_r['test_accuracy'].mean():.4f} +/- {cv_r['test_accuracy'].std():.4f}")

    model.fit(X_train_en, y_train)
    y_p  = model.predict(X_test_en)
    y_pr = model.predict_proba(X_test_en)[:, 1]
    m = {
        "model": name,
        "accuracy":  accuracy_score(y_test, y_p),
        "precision": precision_score(y_test, y_p),
        "recall":    recall_score(y_test, y_p),
        "f1":        f1_score(y_test, y_p),
        "roc_auc":   roc_auc_score(y_test, y_pr),
    }
    print(f"{name} — test set:")
    for k, v in m.items():
        if k != "model":
            print(f"  {k:9s}: {v:.4f}")
    all_results.append(m)

rf = models["random_forest_demographic"]
imp = pd.Series(rf.feature_importances_, index=ENTRY_ENR_COLS).sort_values(ascending=False)
print("\nRandom forest feature importances (top 10, impurity-based —")
print("mechanically favours continuous variables):")
print(imp.head(10).round(3).to_string())

## Block 4b — Isolating the exposure-time confound

Entry year mixes a legitimate cohort-composition effect (late cohorts are
mostly mobile arrivals) with an exposure artifact: individuals entering near
the end of the observation window have mechanically had little time to be
observed departing — the right-censoring bias handled explicitly by the
Kaplan-Meier analysis (NB6). To bound entry year's contribution, the
gradient boosting is retrained without it; the AUC drop is an upper bound on
the combined cohort-plus-exposure signal carried by that single feature. The
departure rate by entry year (also plotted in Block 5) shows three regimes:
a survivorship gradient for old cohorts (left truncation), a peak around
2014-2016, and the exposure cliff after 2022.

In [ ]:
COLS_NO_YEAR = [c for c in ENTRY_ENR_COLS if c != "annee_debut"]

hgb_noyear = HistGradientBoostingClassifier(random_state=RANDOM_STATE)
cv_ny = cross_validate(hgb_noyear, X_train_en[COLS_NO_YEAR], y_train, cv=cv,
                       scoring=["roc_auc", "accuracy"], n_jobs=1)
print("HGB without annee_debut — 5-fold CV (train):")
print(f"  ROC-AUC : {cv_ny['test_roc_auc'].mean():.4f} +/- {cv_ny['test_roc_auc'].std():.4f}")

hgb_noyear.fit(X_train_en[COLS_NO_YEAR], y_train)
auc_ny = roc_auc_score(y_test, hgb_noyear.predict_proba(X_test_en[COLS_NO_YEAR])[:, 1])
auc_hgb = [m for m in all_results if m["model"] == "hgb_demographic"][0]["roc_auc"]
print(f"HGB without annee_debut — test ROC-AUC: {auc_ny:.4f}")
print(f"AUC carried by annee_debut (upper bound): {auc_hgb - auc_ny:+.4f}")

rate_by_year = pop.groupby("annee_debut")["target_depart"].agg(["mean", "size"])
print("\nDeparture rate by entry year:")
print(rate_by_year.round(3).to_string())

## Block 5 — Final comparison table and figures

The comparison table consolidates the test metrics of the five models that
tell the story: entry-only baseline, full entry logistic, the two tree
ensembles, and the leaky entry+trajectory model as a counter-example.
Figure 1 shows the departure rate by entry year (cohorts with n >= 100 to
suppress noisy pre-1947 years, noted in the caption), revealing the three
regimes discussed in Block 4b. Figure 2 compares the ROC curves of the
honest progression against the leaky model.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

# --- Final comparison table (5 rows, no duplicates) ---
results_df = (pd.DataFrame(all_results)
                .drop_duplicates(subset="model", keep="last")
                .set_index("model")
                .loc[["logit_entry", "logit_entry_demographic",
                      "random_forest_demographic", "hgb_demographic",
                      "logit_full_leaky"]]
                .round(4))
print(results_df.to_string())
results_df.to_parquet(DATA_DIR / "nb7_model_comparison.parquet")

# --- Figure 1: departure rate by entry year ---
rby = pop.groupby("annee_debut")["target_depart"].agg(["mean", "size"])
rby = rby[rby["size"] >= 100]
fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(rby.index, rby["mean"], color="tab:blue", lw=2)
ax1.set_xlabel("Entry year (start of first episode)")
ax1.set_ylabel("Departure rate", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue"); ax1.set_ylim(0, 1)
ax1.axvline(2014, color="grey", ls=":", lw=1)
ax1.axvline(2022, color="grey", ls=":", lw=1)
ax1.annotate("snapshot ", xy=(2014, 0.95), ha="right", fontsize=8, color="grey")
ax1.annotate("exposure cliff ", xy=(2022, 0.90), ha="right", fontsize=8, color="grey")
ax2 = ax1.twinx()
ax2.bar(rby.index, rby["size"], alpha=0.15, color="tab:grey")
ax2.set_ylabel("Number of individuals", color="tab:grey")
ax2.tick_params(axis="y", labelcolor="tab:grey")
ax1.set_title("Departure rate by entry year (cohorts with n >= 100)")
fig.tight_layout()
fig.savefig(FIG_DIR / "nb7_departure_rate_by_entry_year.png", dpi=200)
plt.show()

# --- Figure 2: ROC progression vs leaky model ---
hgb = models["hgb_demographic"]
auc_e   = results_df.loc["logit_entry", "roc_auc"]
auc_d   = results_df.loc["logit_entry_demographic", "roc_auc"]
auc_h   = results_df.loc["hgb_demographic", "roc_auc"]
auc_l   = results_df.loc["logit_full_leaky", "roc_auc"]
curves = [
    (f"Entry only (AUC {auc_e:.3f})",            y_proba,    "tab:blue",   "-"),
    (f"+ demographics, logit (AUC {auc_d:.3f})", y_proba_en, "tab:green",  "-"),
    (f"+ demographics, HGB (AUC {auc_h:.3f})",   hgb.predict_proba(X_test_en)[:, 1], "tab:orange", "-"),
    (f"Entry + trajectory, leaky ({auc_l:.3f})", y_proba_f,  "tab:red",    "--"),
]
fig, ax = plt.subplots(figsize=(7, 6))
for label, proba, color, ls in curves:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, label=label, color=color, ls=ls, lw=1.8)
ax.plot([0, 1], [0, 1], color="grey", ls=":", lw=1, label="Random (AUC 0.5)")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC curves, departure prediction (test set)")
ax.legend(loc="lower right", fontsize=9)
fig.tight_layout()
fig.savefig(FIG_DIR / "nb7_roc_comparison.png", dpi=200)
plt.show()

## Block 6a — Error analysis: what the model fails to predict

The strongest test-set errors of the gradient boosting are profiled:
"surprising stayers" (predicted departure probability above 0.8 but present)
and "surprising leavers" (below 0.2 but departed). By construction, what the
model systematically misses is a pattern absent from its features. Each
surprise group is compared to its correctly-predicted counterpart (same
prediction range) on district, permit, household type, entry type and entry
year, to locate where the unexplained retention or departure concentrates.

In [ ]:
hgb = models["hgb_demographic"]
test = pop[~mask_train].copy()
test["proba"] = hgb.predict_proba(X_test_en)[:, 1]

surp_stay  = test[(test["proba"] > 0.8) & (test["target_depart"] == 0)]
conf_leave = test[(test["proba"] > 0.8) & (test["target_depart"] == 1)]
surp_leave = test[(test["proba"] < 0.2) & (test["target_depart"] == 1)]
conf_stay  = test[(test["proba"] < 0.2) & (test["target_depart"] == 0)]
print(f"Surprising stayers : {len(surp_stay):,} vs {len(conf_leave):,} confirmed leavers (p>0.8)")
print(f"Surprising leavers : {len(surp_leave):,} vs {len(conf_stay):,} confirmed stayers (p<0.2)")

def compare(a, b, name):
    print(f"\n=== {name}: surprises vs correctly predicted ===")
    for col in ["quartier", "permis_groupe", "menage_type", "debut_type_premier"]:
        pa = a[col].value_counts(normalize=True).head(4).round(3)
        pb = b[col].value_counts(normalize=True).reindex(pa.index).round(3)
        print(f"\n{col}:  (surprise | correct)")
        for k in pa.index:
            pv = pb[k] if pd.notna(pb[k]) else 0
            print(f"  {str(k):15s} {pa[k]:.3f} | {pv:.3f}")
    print(f"\nannee_debut median: {a['annee_debut'].median():.0f} | {b['annee_debut'].median():.0f}")
    print(f"age_entree median : {a['age_entree'].median():.1f} | {b['age_entree'].median():.1f}")

compare(surp_stay, conf_leave, "Surprising STAYERS")
compare(surp_leave, conf_stay, "Surprising LEAVERS")

## Block 6b — Shallow decision tree as an interaction detector

A depth-4 decision tree is fitted on the same entry features, not as a
competing model but as a readable detector of interactions: its leaves
exhibit subgroups with extreme departure rates defined by combinations of
conditions, which neither the additive logistic coefficients nor the global
clusters can show. Leaves with at least 2,000 training individuals and a
departure rate below 30% or above 70% are reported.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree = DecisionTreeClassifier(max_depth=4, min_samples_leaf=2000,
                              random_state=RANDOM_STATE)
tree.fit(X_train_en, y_train)
print(export_text(tree, feature_names=list(X_train_en.columns),
                  show_weights=True, decimals=1))

leaf_id = tree.apply(X_train_en)
leaves = pd.DataFrame({"leaf": leaf_id, "y": y_train.values}) \
           .groupby("leaf")["y"].agg(["size", "mean"]).round(3)
extreme = leaves[(leaves["size"] >= 2000) & ((leaves["mean"] < 0.3) | (leaves["mean"] > 0.7))]
print(f"\nExtreme leaves (n>=2000, rate <0.3 or >0.7): {len(extreme)}")
print(extreme.sort_values("mean").to_string())